# Stages 5-10 on Kaggle

Settings: **Accelerator = GPU T4** (one GPU is enough), **Internet = On**. Kaggle secrets: `GITHUB_TOKEN`, `HF_TOKEN`, `HF_TENSOR_REPO_ID`.

Run the setup cell, then the stage cells in order. Every stage commits and pushes its outputs to GitHub `main` when it finishes, so if the session dies you can restart the notebook and continue from the next stage (finished stages print `[SKIP]`). Stages 5, 6, 7 and 10 also keep a resumable checkpoint inside a session.

Rough T4 timings: Stage 5 ~25-40 min, Stage 6 ~30-45 min, Stage 7 ~2-3 h (WikiText KL at every position for 366 directions), Stage 8 ~5 min, Stage 9 <1 min, Stage 10 ~15-25 min.

Optional switches (set in the setup cell): `FORCE=1` recomputes finished stages; `SKIP_MMLU=1` applies Day 14 Cut 2 to Stage 7.

In [ ]:
import os, subprocess, sys
from kaggle_secrets import UserSecretsClient

s = UserSecretsClient()
for name in ("GITHUB_TOKEN", "HF_TOKEN", "HF_TENSOR_REPO_ID"):
    os.environ[name] = s.get_secret(name).strip()
os.environ["FORCE"] = "0"      # "1" recomputes stages that already verified
os.environ["SKIP_MMLU"] = "0"  # "1" = Day 14 Cut 2 (WikiText-2 KL only in Stage 7)

REPO = "/kaggle/working/repo"
URL = "https://x-access-token:{}@github.com/DakshUbhadia/unanswerability_to_hallucination.git"


def sh(cmd, cwd=None):
    """Run a shell command, streaming its output into the notebook; raise on failure."""
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {cmd}")


def run_stage(stage):
    """Sync with GitHub main, then run one stage (e.g. 5, '5-9' or 'all') through scripts/kaggle_run.sh."""
    url = URL.format(os.environ["GITHUB_TOKEN"])
    sh(f"git remote set-url origin {url} && git pull -q --ff-only origin main", cwd=REPO)
    sh(f"bash scripts/kaggle_run.sh {stage}", cwd=REPO)


if not os.path.isdir(REPO):
    sh(f"git clone -q {URL.format(os.environ['GITHUB_TOKEN'])} {REPO}")
sh("git log --oneline -3 && nvidia-smi --query-gpu=name,memory.total --format=csv", cwd=REPO)

In [ ]:
# Stage 5: validation sweep over all L x K coordinates, lock (l*, i*), Table 2
run_stage(5)

In [ ]:
# Stage 6: held-out test H1-H3, 119 control directions, generation audit, Tables 1 & 4
run_stage(6)

In [ ]:
# Stage 7: capability retention (WikiText-2 KL for all directions, chat MMLU)
run_stage(7)

In [ ]:
# Stage 8: PopQA H4/H5, Holm-Bonferroni over H1-H5, Figure 2A/S1 data, Table 3
run_stage(8)

In [ ]:
# Stage 9: figures, final tables, README results block (CPU only)
run_stage(9)
from IPython.display import Image, display
for f in ("fig1_intervention_pareto", "fig2a_popqa_violin", "figS1_layerwise_auroc"):
    display(Image(f"{REPO}/figures/{f}.png", width=900))

In [ ]:
# Stage 10 (optional stretch): selective-abstention steering on matched PopQA, Figure 2B; refreshes the README
run_stage(10)
from IPython.display import Image, display
display(Image(f"{REPO}/figures/fig2b_selective_steering.png", width=900))